In [3]:
import pandas as pd

In [4]:
titanic = pd.read_csv('data/Titanic-Dataset.csv')


In [5]:
titanic.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

titanic = pd.read_csv("data/Titanic-Dataset.csv")  
target_col = "Survived"

train_df, test_df = train_test_split(titanic, test_size=0.2, random_state=42, stratify=titanic[target_col])

def inject_leakage(df, target_col, seed=42):
    rng = np.random.default_rng(seed)
    df = df.copy()
    noise = rng.normal(0, 0.05, len(df))  # small noise so corr isn't a suspicious exact 1.0
    df["leaky_survived"] = df[target_col] + noise
    return df

train_leak = inject_leakage(train_df, target_col)
test_leak = inject_leakage(test_df, target_col)


print("leakage corr:", train_leak["leaky_survived"].corr(train_leak[target_col]))


def inject_train_test_duplication(train_df, test_df, frac=0.15, seed=42):
    n_dupes = int(len(test_df) * frac)
    dupe_rows = train_df.sample(n=n_dupes, random_state=seed)
    keep_rows = test_df.sample(n=len(test_df) - n_dupes, random_state=seed)
    contaminated_test = pd.concat([keep_rows, dupe_rows], ignore_index=True)
    ground_truth_leaked_idx = list(range(len(keep_rows), len(contaminated_test)))
    return contaminated_test, ground_truth_leaked_idx

test_dup, dup_ground_truth = inject_train_test_duplication(train_df, test_df)


feature_cols = [c for c in train_df.columns if c != target_col]
n_actual_dupes = test_dup[feature_cols].duplicated(keep=False).sum()
print("planted dupes:", len(dup_ground_truth), "| detected via naive dupe check:", n_actual_dupes)


def inject_class_imbalance(df, target_col, minority_class, keep_frac=0.05, seed=42):
    df = df.copy()
    minority = df[df[target_col] == minority_class].sample(frac=keep_frac, random_state=seed)
    majority = df[df[target_col] != minority_class]
    return pd.concat([majority, minority], ignore_index=True)


minority_class = train_df[target_col].value_counts().idxmin()
train_imbalanced = inject_class_imbalance(train_df, target_col, minority_class)


print("class balance after injection:\n", train_imbalanced[target_col].value_counts(normalize=True))

leakage corr: 0.9949462380465746
planted dupes: 26 | detected via naive dupe check: 0
class balance after injection:
 Survived
0    0.969095
1    0.030905
Name: proportion, dtype: float64


In [7]:
full_leakage_case = pd.concat([train_leak, test_leak], ignore_index=True)
full_leakage_case.to_csv("data/adversarial_leakage.csv", index=False)

full_dup_case = pd.concat([train_df, test_dup], ignore_index=True)
full_dup_case.to_csv("data/adversarial_duplication.csv", index=False)

train_imbalanced.to_csv("data/adversarial_imbalance.csv", index=False)



In [8]:
df=pd.read_csv('data/cancer-DataSet.csv')

In [9]:
df['diagnosis'].value_counts()

diagnosis
B    357
M    212
Name: count, dtype: int64

In [1]:
from autogluon.tabular import TabularDataset, TabularPredictor

c:\Users\User\OneDrive - Nanyang Technological University\Desktop\autonomousMLproject\autonomousML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
train_data = TabularDataset(train_leak)
test_data = TabularDataset(test_leak)

predictor = TabularPredictor(label='Survived').fit(train_data,presets='medium_quality',time_limit=60)
predictions = predictor.predict(test_data)

No path specified. Models will be saved in: "AutogluonModels\ag-20260704_040242"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       1.31 GB / 15.22 GB (8.6%)
Disk Space Avail:   106.82 GB / 926.37 GB (11.5%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 60s
AutoGluon will save models to "c:\Users\User\OneDrive - Nanyang Technological University\Desktop\autonomousMLproject\autonomousML\AutogluonModels\ag-20260704_040242"
Train Data Rows:    712
Train Data Columns: 12
Label Column:       Survived
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label 

In [17]:
lb = predictor.leaderboard(silent=True)   # a DataFrame, best model first
# -> feed into AgentState["leaderbord"] as list[dict]:
predictions = predictor.predict(test_data)

In [18]:
lb

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost,1.0,accuracy,0.005820,16.394048,0.005820,16.394048,1,True,5
1,LightGBMXT,1.0,accuracy,0.006298,0.505920,0.006298,0.505920,1,True,1
2,LightGBMLarge,1.0,accuracy,0.006924,1.220858,0.006924,1.220858,1,True,11
3,LightGBM,1.0,accuracy,0.007187,0.492689,0.007187,0.492689,1,True,2
4,XGBoost,1.0,accuracy,0.010581,3.462392,0.010581,3.462392,1,True,9
5,NeuralNetFastAI,1.0,accuracy,0.015651,4.648732,0.015651,4.648732,1,True,8
6,WeightedEnsemble_L2,1.0,accuracy,0.016658,4.710336,0.001007,0.061603,2,True,12
7,ExtraTreesEntr,1.0,accuracy,0.043025,0.531593,0.043025,0.531593,1,True,7
8,ExtraTreesGini,1.0,accuracy,0.044171,0.552823,0.044171,0.552823,1,True,6
9,RandomForestGini,1.0,accuracy,0.058539,0.705170,0.058539,0.705170,1,True,3


In [20]:
metrics = predictor.evaluate(test_data)

In [21]:
metrics

{'accuracy': 1.0,
 'balanced_accuracy': np.float64(1.0),
 'mcc': 1.0,
 'roc_auc': np.float64(1.0),
 'f1': 1.0,
 'precision': 1.0,
 'recall': 1.0}

In [23]:
train_df, test_dup
train_data = TabularDataset(train_df)
test_data = TabularDataset(test_dup)

predictor = TabularPredictor(label='Survived').fit(train_data,presets='medium_quality',time_limit=60)
predictions = predictor.predict(test_data)
metrics = predictor.evaluate(test_data)


No path specified. Models will be saved in: "AutogluonModels\ag-20260704_040716"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       1.16 GB / 15.22 GB (7.6%)
Disk Space Avail:   106.76 GB / 926.37 GB (11.5%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 60s
AutoGluon will save models to "c:\Users\User\OneDrive - Nanyang Technological University\Desktop\autonomousMLproject\autonomousML\AutogluonModels\ag-20260704_040716"
Train Data Rows:    712
Train Data Columns: 11
Label Column:       Survived
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label 

In [28]:

train_df, test_df = train_test_split(train_imbalanced, test_size=0.2, random_state=42, stratify=train_imbalanced[target_col])


train_data = TabularDataset(train_df)
test_data = TabularDataset(test_df)

predictor = TabularPredictor(label='Survived').fit(train_data,presets='medium_quality',time_limit=60)
predictions = predictor.predict(test_data)
metrics = predictor.evaluate(test_data)


No path specified. Models will be saved in: "AutogluonModels\ag-20260704_041614"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       1.19 GB / 15.22 GB (7.8%)
Disk Space Avail:   106.70 GB / 926.37 GB (11.5%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 60s
AutoGluon will save models to "c:\Users\User\OneDrive - Nanyang Technological University\Desktop\autonomousMLproject\autonomousML\AutogluonModels\ag-20260704_041614"
Train Data Rows:    362
Train Data Columns: 11
Label Column:       Survived
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label 

In [29]:
metrics

{'accuracy': 0.967032967032967,
 'balanced_accuracy': np.float64(0.5000000000000002),
 'mcc': 0.0,
 'roc_auc': np.float64(0.9204545454545454),
 'f1': 0.0,
 'precision': 0.0,
 'recall': 0.0}